# 📈 KOSPI 주식 차트 대시보드

이 노트북은 `11_kospi.py` (Streamlit 앱)을 만들기 전에,
각 구성 요소를 한 셀씩 실행하며 이해하는 **학습용 노트북**입니다.

## 학습 목표
| # | 주제 | 핵심 함수/개념 |
|---|------|----------------|
| 1 | 라이브러리 설치 | `uv add`, `pip install` |
| 2 | 시장 전체 종목 조회 | `fdr.StockListing()` |
| 3 | 한글 정규화 | `unicodedata.normalize()` |
| 4 | 시가총액 TOP10 추출 | `.nlargest()`, `.iloc[::-1]` |
| 5 | 수평 막대그래프 | `go.Bar(orientation='h')` |
| 6 | 개별 종목 데이터 조회 | `fdr.DataReader()` |
| 7 | 종가 라인 차트 | `px.line()` |
| 8 | 다중 종목 병합 | `pd.concat(axis=1)` |
| 9 | 캔들스틱 차트 | `go.Candlestick()` |

---

## STEP 0 — 라이브러리 설치

> Streamlit 프로젝트에서는 `uv add`로 설치합니다.  
> Colab / 노트북 환경에서는 아래 셀을 실행하세요.

In [ ]:
# Colab / 노트북 환경 설치
# Streamlit 프로젝트라면: uv add finance-datareader plotly
# !pip install finance-datareader plotly -q

https://financedata.github.io/posts/finance-data-reader-users-guide.html

## STEP 1 — 라이브러리 임포트

In [1]:
import FinanceDataReader as fdr
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import unicodedata
import datetime

print('✅ 라이브러리 임포트 완료')

✅ 라이브러리 임포트 완료


## STEP 2 — KOSPI 전체 종목 리스트 조회

```python
fdr.StockListing(market)
```
- 인자로 시장 코드를 넣으면 해당 시장의 **전체 종목 DataFrame**을 반환
- 주요 시장 코드: `'KOSPI'`, `'KOSDAQ'`, `'NYSE'`, `'NASDAQ'`
- 주요 컬럼: `Code`(종목코드), `Name`(종목명), `Marcap`(시가총액), `Sector`(업종)

In [2]:
market = 'KOSPI'
df_market = fdr.StockListing(market)

print(f'전체 종목 수: {len(df_market)}개')
print(f'컬럼 목록: {df_market.columns.tolist()}')
df_market.head()

전체 종목 수: 948개
컬럼 목록: ['Unnamed: 0', 'Code', 'ISU_CD', 'Name', 'Market', 'Dept', 'Close', 'ChangeCode', 'Changes', 'ChagesRatio', 'Open', 'High', 'Low', 'Volume', 'Amount', 'Marcap', 'Stocks', 'MarketId']


,Unnamed: 0,Code,ISU_CD,Name,Market,Dept,Close,ChangeCode,Changes,ChagesRatio,Open,High,Low,Volume,Amount,Marcap,Stocks,MarketId
0,0,005930,KR7005930003,삼성전자,KOSPI,NaN,268500,2,-3000,-1.10,260000,270000,260000,25875880,6856898737478,1569725806248000,5846278608,STK
1,1,000660,KR7000660001,SK하이닉스,KOSPI,NaN,1686000,1,32000,1.93,1591000,1689000,1591000,4278087,7029929370772,1201616187390000,712702365,STK
2,2,005935,KR7005931001,삼성전자우,KOSPI,NaN,182700,2,-2700,-1.46,176000,183700,176000,5449257,977214055438,146593218788100,802371203,STK
3,3,402340,KR7402340004,SK스퀘어,KOSPI,NaN,1098000,2,-1000,-0.09,1048000,1106000,1047000,720073,779449679500,144890307828000,131958386,STK
4,4,005380,KR7005380001,현대차,KOSPI,NaN,613000,1,41000,7.17,590000,647000,581000,5020568,3104780887000,125516510558000,204757766,STK


## STEP 3 — 한글 종목명 정규화 (unicodedata)

### 왜 필요한가?
FinanceDataReader에서 가져온 종목명과 사용자가 입력한 문자열의 **유니코드 인코딩이 미묘하게 달라**서  
`==` 비교 시 같은 글자처럼 보여도 `False`가 되는 경우가 있습니다.

```
'삼성전자' (NFC)  ≠  '삼성전자' (NFD)  →  normalize로 통일 필요
```

### NFKC 방식이란?
- **K** (Compatibility Decomposition): 전각/반각 등 호환 문자를 분해
- **C** (Canonical Composition): 정준 방식으로 다시 결합
- 예) `'㈜'` → `'(주)'`, `'Ａ'`(전각) → `'A'`(반각)

In [3]:
# 정규화 전후 비교 예시
# s1 = '삼성전자'           # NFC (일반적인 한글 입력)
# s2 = '\u삼\u성\u전\u자'  # NFD (분해된 형태, 눈에는 같아 보임)

def normalize_str(s):
    return unicodedata.normalize('NFKC', s).strip()

# 전각 문자 예시 (실제로 종목명에서 발생할 수 있는 케이스)
test_cases = [
    ('㈜카카오', '정규화 전'),
    (normalize_str('㈜카카오'), '정규화 후'),
]

for text, label in test_cases:
    print(f'{label}: {text}')

# 실제 적용: df_market 종목명 전체 정규화
df_market['Name'] = df_market['Name'].apply(normalize_str)
print('\n✅ 종목명 정규화 완료')
df_market[['Code', 'Name', 'Marcap']].head()

정규화 전: ㈜카카오
정규화 후: (주)카카오

✅ 종목명 정규화 완료


,Code,Name,Marcap
0,005930,삼성전자,1569725806248000
1,000660,SK하이닉스,1201616187390000
2,005935,삼성전자우,146593218788100
3,402340,SK스퀘어,144890307828000
4,005380,현대차,125516510558000


## STEP 4 — 시가총액 TOP 10 추출

```python
df.nlargest(n, column)   # 특정 컬럼 기준 상위 n개 행 추출
df.iloc[::-1]            # 행 순서를 뒤집기 (역순 정렬)
```

> `iloc[::-1]` 이 필요한 이유:  
> 수평 막대그래프(`orientation='h'`)는 **아래에서 위로** 값을 쌓기 때문에,  
> 역순으로 뒤집어야 **위쪽에 1위**가 표시됩니다.

In [6]:
top10 = df_market.nlargest(10, 'Marcap').iloc[::-1]

# 시가총액 단위 변환: 원 → 조 (1조 = 10^12)
top10_display = top10[['Name', 'Marcap']].copy()
top10_display['시가총액(조)'] = (top10_display['Marcap'] / 1e12).round(1)

top10_display[['Name', '시가총액(조)']].reset_index(drop=True)

,Name,시가총액(조)
0,삼성전기,68.3
1,삼성물산,68.5
2,HD현대중공업,69.1
3,두산에너빌리티,83.0
4,LG에너지솔루션,111.5
5,현대차,125.5
6,SK스퀘어,144.9
7,삼성전자우,146.6
8,SK하이닉스,1201.6
9,삼성전자,1569.7


## STEP 5 — 시가총액 TOP10 수평 막대그래프 (Plotly)

```python
go.Bar(
    orientation='h',       # 'h': 수평 / 'v': 수직(기본값)
    text=...,              # 막대 위에 텍스트 표시
    texttemplate='...'     # 텍스트 포맷 지정
)
```

In [7]:
fig_top10 = go.Figure(go.Bar(
    x=top10['Marcap'] / 1e12,
    y=top10['Name'],
    orientation='h',                        # 수평 막대그래프
    text=top10['Marcap'] / 1e12,
    texttemplate='%{text:.1f}조',           # 소수점 1자리 + '조' 단위
    marker_color='steelblue'                # 막대 색상
))

fig_top10.update_layout(
    title=f'{market} 시가총액 TOP10',
    xaxis_title='시가총액 (조)',
    yaxis_title='종목명',
    bargap=0.15,                            # 막대 사이 간격 (0~1)
    height=450
)

fig_top10.show()

## STEP 6 — 개별 종목 주가 데이터 조회

```python
fdr.DataReader(code, start, end)
```
- `code`: 종목 코드 (예: `'005930'` = 삼성전자)
- `start`, `end`: 날짜 문자열 `'YYYY-MM-DD'` 형식
- 반환 컬럼: `Open`(시가), `High`(고가), `Low`(저가), `Close`(종가), `Volume`(거래량)

> **종목명 → 종목코드 변환**이 필요합니다.  
> `fdr.DataReader()`는 종목명이 아닌 **코드**를 입력받습니다.

In [9]:
# 종목명 → 종목코드 변환 예시
target_name = '삼성전자'
code = df_market.loc[df_market['Name'] == target_name, 'Code'].values[0]
print(f'{target_name} 종목코드: {code}')

# 주가 데이터 조회
start_date = '2024-01-01'
end_date   = datetime.datetime.now().strftime('%Y-%m-%d')

df_stock = fdr.DataReader(code, start_date, end_date)

print(f'\n조회 기간: {start_date} ~ {end_date}')
print(f'데이터 수: {len(df_stock)}일')
df_stock.tail()

삼성전자 종목코드: 005930

조회 기간: 2024-01-01 ~ 2026-05-10
데이터 수: 571일


,Open,High,Low,Close,Volume,Change
Date,,,,,,
2026-04-30,229000,230000,220500,220500,22161975,-0.024336
2026-05-04,228000,232500,224000,232500,32920816,0.054422
2026-05-06,254000,270000,251000,266000,53097996,0.144086
2026-05-07,272000,277000,260000,271500,41404687,0.020677
2026-05-08,260000,270000,260000,268500,25696964,-0.011050


## STEP 7 — 현재가 / 전일 대비 확인

```python
df['Close'].iloc[-1]   # 가장 최근 종가 (마지막 행)
df['Close'].iloc[-2]   # 전일 종가 (마지막-1 행)
```

> Streamlit에서는 `st.metric()`으로 카드 형태로 표시합니다.  
> 노트북에서는 print로 확인합니다.

In [10]:
current = df_stock['Close'].iloc[-1]   # 최근 종가
prev    = df_stock['Close'].iloc[-2]   # 전일 종가
delta   = current - prev               # 전일 대비 변동폭
pct     = delta / prev * 100           # 등락률 (%)

direction = '▲' if delta > 0 else '▼'

print(f'[ {target_name} ({code}) ]')
print(f'  현재가   : {current:,}원')
print(f'  전일 대비: {direction} {abs(delta):,}원  ({pct:+.2f}%)')

[ 삼성전자 (005930) ]
  현재가   : 268,500원
  전일 대비: ▼ 3,000원  (-1.10%)


## STEP 8 — 종가 라인 차트 (단일 종목)

In [11]:
fig_line = px.line(
    df_stock,
    y='Close',
    title=f'{target_name} 종가 추이',
    labels={'Close': '종가 (원)', 'Date': '날짜'}
)

fig_line.update_layout(height=400)
fig_line.show()

## STEP 9 — 다중 종목 종가 비교 (pd.concat)

여러 종목의 종가를 **날짜 인덱스 기준으로 수평 병합**하면 한 번에 비교할 수 있습니다.

```python
pd.concat([df1, df2, df3], axis=1)   # axis=1: 수평(열 방향) 병합
                                      # axis=0: 수직(행 방향) 이어붙임 (기본값)
```

병합 결과 예시:
```
            삼성전자    SK하이닉스
2024-01-02   78000      130000
2024-01-03   77500      128000
```

In [12]:
# 비교할 종목 목록
compare_names = ['삼성전자', 'SK하이닉스', 'LG에너지솔루션']

dfs = []
for name in compare_names:
    matched = df_market.loc[df_market['Name'] == name, 'Code'].values
    if len(matched) == 0:
        print(f'⚠️ {name}: 종목코드를 찾을 수 없습니다.')
        continue
    
    c = matched[0]
    df_temp = fdr.DataReader(c, start_date, end_date)
    
    if not df_temp.empty:
        # Close 열만 추출하고, 열 이름을 종목명으로 변경
        dfs.append(df_temp[['Close']].rename(columns={'Close': name}))
        print(f'✅ {name} ({c}) 데이터 로드 완료 ({len(df_temp)}일)')

# 수평 병합 (날짜 인덱스 기준 자동 정렬)
merged_df = pd.concat(dfs, axis=1)
print(f'\n병합 결과: {merged_df.shape}')
merged_df.tail()

✅ 삼성전자 (005930) 데이터 로드 완료 (571일)
✅ SK하이닉스 (000660) 데이터 로드 완료 (571일)
✅ LG에너지솔루션 (373220) 데이터 로드 완료 (571일)

병합 결과: (571, 3)


,삼성전자,SK하이닉스,LG에너지솔루션
Date,,,
2026-04-30,220500,1286000,460500
2026-05-04,232500,1447000,472000
2026-05-06,266000,1601000,482000
2026-05-07,271500,1654000,483000
2026-05-08,268500,1686000,476500


In [13]:
# 다중 종목 라인 차트
fig_multi = px.line(
    merged_df,
    title='주요 종목 종가 비교',
    labels={'value': '종가 (원)', 'Date': '날짜', 'variable': '종목'}
)

fig_multi.update_layout(height=450, legend_title='종목')
fig_multi.show()

## STEP 10 — 캔들스틱 차트 (go.Candlestick)

캔들스틱은 하루의 주가 흐름을 **시가·고가·저가·종가** 4가지로 표현합니다.

```
  ─  고가 (High)
  █  양봉: 종가 > 시가 → 주가 상승 (초록)
  █  음봉: 종가 < 시가 → 주가 하락 (빨강)
  ─  저가 (Low)
```

```python
go.Candlestick(
    x=df.index,       # x축: 날짜
    open=df['Open'],  # 시가
    high=df['High'],  # 고가
    low=df['Low'],    # 저가
    close=df['Close'] # 종가
)
```

In [ ]:
# 최근 3개월만 표시 (캔들이 너무 많으면 보기 어려움)
three_months_ago = df_stock.index[-1] - pd.DateOffset(months=3)
df_candle = df_stock[df_stock.index >= three_months_ago]

# df_stock.index[-1]
# DataFrame의 인덱스(날짜)에서 마지막 값, 즉 가장 최근 날짜를 가져옵니다.
# 예) 2026-05-09

# pd.DateOffset(months=3)
# 날짜 연산을 위한 pandas 객체입니다. 3M 처럼 달력 기준으로 정확히 3개월을 빼줍니다.
# datetime.timedelta(days=90) 과 비슷하지만, 월말 처리 등 달력 기준으로 더 정확합니다.


fig_candle = go.Figure(data=[go.Candlestick(
    x=df_candle.index,
    open=df_candle['Open'],
    high=df_candle['High'],
    low=df_candle['Low'],
    close=df_candle['Close']
)])

# df_stock[df_stock.index >= three_months_ago]
# 인덱스(날짜)가 3개월 전 이후인 행만 필터링합니다. pandas 불리언 인덱싱입니다.

# df_stock.index >= three_months_ago
# # → [False, False, ..., True, True, True]  ← 최근 3개월만 True

# df_stock[조건]  # True인 행만 남김


fig_candle.update_layout(
    title=f'{target_name} 캔들스틱 차트 (최근 3개월)',
    xaxis_title='날짜',
    yaxis_title='가격 (원)',
    xaxis_rangeslider_visible=False,   # 하단 범위 슬라이더 숨김
    height=500
)

fig_candle.show()

## STEP 11 — 종합 정리: Streamlit 앱과의 매핑

| 노트북 STEP | Streamlit 앱 섹션 | 핵심 포인트 |
|------------|------------------|-------------|
| STEP 2 | 섹션 3: 종목 리스트 로드 | `fdr.StockListing()` |
| STEP 3 | 섹션 2: normalize 함수 | `unicodedata.normalize('NFKC')` |
| STEP 4~5 | 섹션 4: TOP10 차트 | `.nlargest()`, `go.Bar(orientation='h')` |
| STEP 6 | 섹션 8: 데이터 로드 함수 | `fdr.DataReader()` |
| STEP 7 | 섹션 9: metric 카드 | `.iloc[-1]`, `.iloc[-2]` |
| STEP 8~9 | 섹션 10: Tab1 라인 차트 | `st.line_chart()`, `pd.concat(axis=1)` |
| STEP 10 | 섹션 10: Tab2 캔들스틱 | `go.Candlestick()` |

---

## ✅ 다음 단계

이 노트북의 내용을 이해했다면, `11_kospi.py`를 열고 각 섹션이 어떻게 연결되는지 확인해보세요.

```bash
# 프로젝트 폴더에서
source .venv/Scripts/activate
streamlit run 11_kospi.py
```